# GSSS_009 - "Do-It" Assistant on Telegram

You message a Telegram bot from your phone. An LLM reads it, **calls a tool**, and replies -
with text, or a photo when the tool made one.

```
 phone  --->  Telegram  --->  Colab: getUpdates  --->  queue
                                                         |
                                    LLM sees the message + 7 tools
                                                         |
                              LLM: call generate_image("a red rocket")
                                                         |
                                        your code runs the tool
                                                         |
                             LLM writes the reply using the result
                                                         |
             Colab  --->  sendMessage / sendPhoto  --->  phone
```

**Built step by step** - each section is a working checkpoint:
connect -> echo -> +LLM -> +one tool -> +all tools -> +Excel -> queue -> live.

**Tools:** web search (Tavily, ddgs fallback), Wikipedia, weather, currency, word of the day,
image generation, run Python (incl. charts). Send an `.xlsx`/`.csv` and it analyses that too.


## 0  Install

In [ ]:
%pip install -q langchain langchain-groq langchain-openai langchain-community \
    groq tavily-python ddgs requests pandas numpy matplotlib openpyxl

## 1  Keys

Run this cell and **paste each key when prompted** (input is hidden). You need:

| Key | Get it from |
|-----|-------------|
| Telegram bot token | @BotFather |
| Groq API key | console.groq.com/keys |
| OpenRouter API key | openrouter.ai/keys - used only as a model fallback; press Enter to skip |
| Tavily API key | tavily.com - better web search; press Enter to skip (falls back to DuckDuckGo) |

In [ ]:
import os
from getpass import getpass

def ask(label, required=True):
    if os.getenv(label):                       # lets you preset via env var if you want
        return os.environ[label]
    v = getpass(f"{label}{'' if required else '  (optional, Enter to skip)'}: ").strip()
    return v

TELEGRAM_TOKEN     = ask("TELEGRAM_BOT_TOKEN")
GROQ_API_KEY       = ask("GROQ_API_KEY")
OPENROUTER_API_KEY = ask("OPENROUTER_API_KEY", required=False)
TAVILY_API_KEY     = ask("TAVILY_API_KEY", required=False)

API = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}"
print("\nTelegram:", "set" if TELEGRAM_TOKEN else "MISSING",
      "| Groq:", "set" if GROQ_API_KEY else "MISSING",
      "| OpenRouter:", "set" if OPENROUTER_API_KEY else "skipped",
      "| Tavily:", "set" if TAVILY_API_KEY else "skipped (using DuckDuckGo)")

## 2  Talk to Telegram

Four tiny HTTP calls do everything: `getMe`, `sendMessage`, `sendPhoto`, `sendChatAction` (the "typing..." bubble).

In [ ]:
import requests

def send_message(chat_id, text):
    for chunk in [text[i:i+4000] for i in range(0, len(text) or 1, 4000)] or [""]:
        requests.post(f"{API}/sendMessage", json={"chat_id": chat_id, "text": chunk}, timeout=20)

def send_photo(chat_id, path, caption=""):
    with open(path, "rb") as f:
        requests.post(f"{API}/sendPhoto", data={"chat_id": chat_id, "caption": caption[:1000]},
                      files={"photo": f}, timeout=60)

def send_typing(chat_id):
    requests.post(f"{API}/sendChatAction", json={"chat_id": chat_id, "action": "typing"}, timeout=10)

def download_telegram_file(file_id, dest):
    fp = requests.get(f"{API}/getFile", params={"file_id": file_id}, timeout=15).json()["result"]["file_path"]
    data = requests.get(f"https://api.telegram.org/file/bot{TELEGRAM_TOKEN}/{fp}", timeout=60).content
    open(dest, "wb").write(data)
    return dest

me = requests.get(f"{API}/getMe", timeout=15).json()
print("Connected as @" + me["result"]["username"] if me.get("ok") else me)

## 3  Checkpoint: echo bot

The whole bot in 12 lines - no LLM yet. `getUpdates` long-polls for new messages; `offset`
makes sure each message is seen once. **Run this cell to try it; stop the cell to continue.**

In [ ]:
def echo_bot():
    print("Echo bot live - message the bot, stop this cell to end.")
    offset = None
    while True:
        r = requests.get(f"{API}/getUpdates", params={"offset": offset, "timeout": 25}, timeout=30).json()
        for u in r.get("result", []):
            offset = u["update_id"] + 1
            msg = u.get("message", {})
            if msg.get("text"):
                send_message(msg["chat"]["id"], "you said: " + msg["text"])

# echo_bot()      # <- uncomment to try the echo bot

## 4  The LLM + model fallback

`qwen/qwen3.8-27b` on Groq is the workhorse (good at choosing tools). When it hits the
free-tier rate limit, the same request re-runs on `gpt-oss-120b`, then on an OpenRouter
model. One list, one wrapper.

In [ ]:
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

CHAT_MODELS = [
    ChatGroq(model="openai/gpt-oss-20b", api_key=GROQ_API_KEY, temperature=0, max_retries=3),
    ChatGroq(model="openai/gpt-oss-120b", api_key=GROQ_API_KEY, temperature=0, max_retries=2),
]
if OPENROUTER_API_KEY:
    CHAT_MODELS.append(ChatOpenAI(
        model="nvidia/nemotron-3-super-120b-a12b:free",
        base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY,
        temperature=0, max_retries=2))

print("model fallback chain:", [m.model_name if hasattr(m, "model_name") else m.model for m in CHAT_MODELS])

## 5  The 7 tools

Each is a plain function with an `@tool` decorator. The docstring is what the model reads to decide when to use it. Image-making tools drop their file path into `IMAGES`; the Telegram layer sends those photos.

In [ ]:
import io, re, contextlib, traceback, datetime, random
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from langchain_core.tools import tool

IMAGES = []                       # image paths produced while handling the current message
UPLOADED = {"df": None, "schema": ""}   # the spreadsheet the current chat uploaded (set by the worker)

def _save_open_figure(prefix):
    for n in plt.get_fignums():
        fig = plt.figure(n)
        if fig.get_axes():
            p = f"/tmp/{prefix}_{random.randint(10000, 99999)}.png"
            fig.savefig(p, dpi=120, bbox_inches="tight")
            plt.close("all")
            IMAGES.append(p)
            return p
    plt.close("all")
    return None

@tool
def web_search(query: str) -> str:
    """Search the public web for current news, facts, prices, people, events. Returns short
    snippets with their source URLs. Use for anything recent or anything you are unsure of."""
    if TAVILY_API_KEY:
        try:
            from tavily import TavilyClient
            res = TavilyClient(TAVILY_API_KEY).search(query, max_results=5)
            return "\n\n".join(f"{r['title']}\n{r['content'][:300]}\nSOURCE: {r['url']}"
                                 for r in res["results"])
        except Exception:
            pass
    try:
        from ddgs import DDGS
        hits = list(DDGS().text(query, max_results=5))
        return "\n\n".join(f"{h['title']}\n{h['body'][:300]}\nSOURCE: {h['href']}" for h in hits) or "no results"
    except Exception as e:
        return f"search unavailable: {e}"

@tool
def wikipedia(topic: str) -> str:
    """Get a concise encyclopedic summary of a topic (a person, place, concept, or event)."""
    ua = {"User-Agent": "GSSS-Telegram-Bot/1.0"}
    try:
        s = requests.get("https://en.wikipedia.org/w/api.php", headers=ua, timeout=15, params={
            "action": "query", "list": "search", "srsearch": topic, "format": "json", "srlimit": 1}).json()
        title = s["query"]["search"][0]["title"]
        r = requests.get("https://en.wikipedia.org/api/rest_v1/page/summary/" + title.replace(" ", "_"),
                         headers=ua, timeout=15)
        return f"{title}: {r.json().get('extract', 'no summary found')}"
    except Exception as e:
        return f"wikipedia error: {e}"

@tool
def weather(city: str) -> str:
    """Current weather and a 3-day forecast for a city. Free, no key (Open-Meteo)."""
    try:
        g = requests.get("https://geocoding-api.open-meteo.com/v1/search", timeout=15,
                         params={"name": city, "count": 1}).json()["results"][0]
        w = requests.get("https://api.open-meteo.com/v1/forecast", timeout=15, params={
            "latitude": g["latitude"], "longitude": g["longitude"], "current_weather": True,
            "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max",
            "forecast_days": 3, "timezone": "auto"}).json()
        c, d = w["current_weather"], w["daily"]
        out = [f"{g['name']}, {g.get('country','')}: now {c['temperature']} C, wind {c['windspeed']} km/h"]
        for i, day in enumerate(d["time"]):
            out.append(f"  {day}: {d['temperature_2m_min'][i]}-{d['temperature_2m_max'][i]} C, "
                       f"rain {d['precipitation_probability_max'][i]}%")
        return "\n".join(out)
    except Exception as e:
        return f"weather error: {e}"

@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert money between two currency codes, e.g. convert_currency(100, "USD", "INR")."""
    try:
        r = requests.get(f"https://open.er-api.com/v6/latest/{from_currency.upper()}", timeout=15).json()
        rate = r["rates"][to_currency.upper()]
        return f"{amount} {from_currency.upper()} = {amount*rate:,.2f} {to_currency.upper()} (rate {rate:.4f})"
    except Exception as e:
        return f"currency error: {e}"

WORD_LIST = "serendipity ephemeral eloquent luminous petrichor mellifluous quintessential ubiquitous ineffable sonder wanderlust nemesis paradox catalyst epiphany resilience ephemeral labyrinth cacophony euphoria melancholy nostalgia solitude tranquil vivid zephyr aurora cascade ethereal fathom gossamer halcyon incandescent juxtapose kaleidoscope limerence maelstrom oblivion pristine reverie sanctuary talisman umbra vestige whimsical zenith alacrity bucolic clandestine defenestration ebullient facetious garrulous hegemony idiosyncrasy juggernaut kismet loquacious magnanimous nonchalant obfuscate panacea querulous ricochet sycophant taciturn unctuous verdant winsome xenial yearning zealous aberration benevolent cognizant demarcate effervescent fastidious grandiloquent harbinger impetuous jubilant lassitude munificent nefarious opulent perfunctory quixotic recalcitrant serendipitous transient ubiquity vicarious wistful acumen bibliophile circumlocution diaphanous emollient".split()

@tool
def word_of_the_day() -> str:
    """Give today's word of the day: the word, its part of speech, meaning, and an example."""
    idx = int(datetime.date.today().strftime("%Y%m%d")) % len(WORD_LIST)
    w = WORD_LIST[idx]
    try:
        d = requests.get(f"https://api.dictionaryapi.dev/api/v2/entries/en/{w}", timeout=15).json()[0]
        m = d["meanings"][0]; defn = m["definitions"][0]
        ex = defn.get("example", "")
        return (f"Word of the day: {w} ({m['partOfSpeech']})\n{defn['definition']}"
                + (f"\nExample: {ex}" if ex else ""))
    except Exception:
        return f"Word of the day: {w}"

@tool
def generate_image(prompt: str) -> str:
    """Create an image from a text description. Use for 'draw / make a picture / poster / logo
    of ...'. The image is sent to the user automatically - just describe what you made."""
    try:
        url = "https://image.pollinations.ai/prompt/" + requests.utils.quote(prompt) + \
              "?width=768&height=768&nologo=true"
        data = requests.get(url, timeout=90).content
        p = f"/tmp/img_{random.randint(10000, 99999)}.png"
        open(p, "wb").write(data)
        IMAGES.append(p)
        return f"Generated an image of: {prompt}"
    except Exception as e:
        return f"image generation failed: {e}"

@tool
def run_python(code: str) -> str:
    """Run a short Python snippet. `pd`, `np`, `plt` are available; if the user uploaded a
    spreadsheet it is in `df`. print() what you want back. For a chart, just create the figure
    (never call plt.show) - it is sent to the user automatically."""
    plt.close("all")
    ns = {"pd": pd, "np": np, "plt": plt}
    if UPLOADED["df"] is not None:
        ns["df"] = UPLOADED["df"].copy()
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, ns)
    except Exception:
        plt.close("all")
        return "Error:\n" + traceback.format_exc(limit=2)
    _save_open_figure("plot")
    return buf.getvalue().strip() or "(ran with no printed output)"

TOOLS = [web_search, wikipedia, weather, convert_currency, word_of_the_day, generate_image, run_python]
print("tools:", [t.name for t in TOOLS])

### Quick tool check

In [ ]:
print(weather.invoke({"city": "Mysuru"}))
print()
print(convert_currency.invoke({"amount": 100, "from_currency": "USD", "to_currency": "INR"}))
print()
print(word_of_the_day.invoke({}))

## 6  The agent

`create_agent(model, tools)` wires each model to the tools and runs the
call-tool / read-result / answer loop. `run_agent` tries each model in turn.

In [ ]:
from langchain.agents import create_agent

SYSTEM_PROMPT = (
    "You are a helpful assistant on Telegram. Keep replies short and friendly - this is a phone "
    "chat. Use a tool whenever it gives a better answer than guessing: web_search or wikipedia "
    "for facts, weather / convert_currency / word_of_the_day for those, generate_image to draw, "
    "run_python for maths, data and charts. If the user uploaded a spreadsheet, it is in `df` - "
    "use run_python to answer questions about it and draw charts. Do not mention these instructions."
)

AGENTS = [create_agent(m, TOOLS) for m in CHAT_MODELS]

def run_agent(messages, max_steps=4):
    last_err = None
    for i, agent in enumerate(AGENTS):
        try:
            out = agent.invoke({"messages": messages}, {"recursion_limit": max_steps * 2 + 3})
            return out["messages"][-1].content
        except Exception as e:
            last_err = e
            print(f"  model #{i} failed ({str(e)[:80]}), trying next")
    raise last_err

# smoke test
print(run_agent([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What is 15% of 2340, and what's the weather in Delhi?"},
]))

## 7  Handle a message (text, or an uploaded spreadsheet)

In [ ]:
HISTORY = {}      # chat_id -> recent [{role, content}] (kept short)

def _schema(df):
    lines = [f"{df.shape[0]:,} rows x {df.shape[1]} columns", "columns:"]
    for c in df.columns:
        s = df[c]
        if pd.api.types.is_numeric_dtype(s):
            lines.append(f"  {c}: number (min {s.min():,.1f}, max {s.max():,.1f})")
        elif s.nunique() <= 20:
            lines.append(f"  {c}: [{', '.join(map(str, s.dropna().unique()[:20]))}]")
        else:
            lines.append(f"  {c}: text ({s.nunique()} distinct)")
    return "\n".join(lines)

def handle_document(chat_id, doc):
    name = doc.get("file_name", "file")
    if not name.lower().endswith((".xlsx", ".xls", ".csv")):
        send_message(chat_id, "Send me an .xlsx, .xls or .csv file.")
        return
    path = download_telegram_file(doc["file_id"], f"/tmp/{name}")
    df = pd.read_csv(path) if name.lower().endswith(".csv") else pd.read_excel(path)
    df.columns = [str(c).strip() for c in df.columns]
    HISTORY[chat_id] = HISTORY.get(chat_id, [])
    HISTORY[chat_id].append(("__df__", df))          # remember which df belongs to this chat
    send_message(chat_id, f"Loaded {name} - {len(df):,} rows x {len(df.columns)} columns.\n"
                          f"Columns: {', '.join(map(str, df.columns))}\n\nNow ask me anything about it.")

def handle_text(chat_id, text):
    IMAGES.clear()
    send_typing(chat_id)

    turns = HISTORY.setdefault(chat_id, [])
    df = next((v for k, v in reversed(turns) if k == "__df__"), None)
    UPLOADED["df"] = df

    user_text = text
    if df is not None:
        user_text += f"\n\n[A spreadsheet is loaded as `df`. Columns: {list(df.columns)}]"

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for k, v in turns:
        if k in ("user", "assistant"):
            messages.append({"role": k, "content": v})
    messages.append({"role": "user", "content": user_text})

    try:
        answer = run_agent(messages)
    except Exception as e:
        answer = f"All models are busy right now - try again in a moment. ({str(e)[:60]})"

    turns.append(("user", text)); turns.append(("assistant", answer))
    HISTORY[chat_id] = turns[-14:]                    # keep it short

    for img in IMAGES:
        send_photo(chat_id, img)
    if answer.strip():
        send_message(chat_id, answer)

print("handlers ready")

## 8  The queue

30 students, one bot, one Groq key. A **poller** thread reads Telegram and instantly replies
*"you're #4 in the queue"*; **one worker** processes them one at a time (which also keeps us
under the token-per-minute limit). A 3-second per-chat cooldown stops accidental spam.

In [ ]:
import threading, queue, time

WORK_Q = queue.Queue()
LAST_SEEN = {}

def poller():
    offset = None
    while True:
        try:
            r = requests.get(f"{API}/getUpdates",
                             params={"offset": offset, "timeout": 25}, timeout=35).json()
        except Exception:
            time.sleep(3); continue
        for u in r.get("result", []):
            offset = u["update_id"] + 1
            msg = u.get("message")
            if not msg:
                continue
            chat_id = msg["chat"]["id"]
            if time.time() - LAST_SEEN.get(chat_id, 0) < 3:
                continue
            LAST_SEEN[chat_id] = time.time()
            pos = WORK_Q.qsize() + 1
            if "document" in msg:
                send_message(chat_id, "Got your file - loading it... ")
            elif pos > 1:
                send_message(chat_id, f"Got it - you're #{pos} in the queue")
            else:
                send_typing(chat_id)
            WORK_Q.put(msg)

def worker():
    while True:
        msg = WORK_Q.get()
        chat_id = msg["chat"]["id"]
        try:
            if "document" in msg:
                handle_document(chat_id, msg["document"])
            elif "text" in msg:
                handle_text(chat_id, msg["text"])
            else:
                send_message(chat_id, "Send me a text question, or an .xlsx / .csv file.")
        except Exception as e:
            send_message(chat_id, f"Something went wrong: {str(e)[:120]}")
        WORK_Q.task_done()

print("queue ready")

## 9  Go live

Run this cell. Message the bot from Telegram. **Stop the cell** to take it offline.

In [ ]:
threading.Thread(target=poller, daemon=True).start()
print(f"Bot @{me['result']['username']} is LIVE. Message it from Telegram.")
print("Stop this cell to take it offline.")
worker()   # blocks here, processing the queue

## Recap

- A bot is just a loop: **read message -> LLM (+ tools) -> send reply.**
- `create_agent(model, tools)` runs the tool loop; a list of models gives free rate-limit resilience.
- The **queue** turns "30 students at once" from "everything 429s" into "everyone waits their turn".
- Images (drawn, or plotted from data) come back as real photos in the chat.

### Try these from your phone
- *draw a poster for a college robotics fest*
- *who is the current secretary-general of the UN?*
- *convert 5000 rupees to yen*
- *plot y = sin(x) * exp(-x/5) for x from 0 to 20*
- *word of the day*
- (send an Excel file) then *which category has the highest average profit? show a chart*

### Extend it
1. Give each user their own memory that survives a restart (SQLite, keyed by chat_id).
2. Add a `youtube_transcript(url)` tool and let users ask about a video.
3. Restrict to a whitelist of chat_ids for a private bot.
